# Study 886 — Agency MBS Carry 🏠

**Do agency mortgage bonds pay you a real spread over duration-matched Treasuries?**

A mortgage pass-through (MBB, VMBS) is a Treasury bond **plus a short refinancing option**:
homeowners refi when rates fall and sit tight when they rise, so the bond is **negatively
convex** and pays an **option-adjusted spread** as compensation. The carry story says: buy
MBS, duration-hedge with Treasuries (IEF), and pocket that spread. We harvest it as the
duration-neutral, cash-neutral monthly spread on the live ETF tape (2007-06-30 → 2026-06-30).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `82b3eede7f92`);
the live cells run the fast synthetic control. Short-history caveat: this tape is one rate
cycle plus the 2013 / 2020 / 2022 vol shocks — not many independent draws.*


## 1. The idea in one picture

You lend a homeowner money at a fixed rate. If rates **fall**, he refinances and hands your money back — right when you'd have wanted to keep earning the old high rate. If rates **rise**, he keeps the cheap mortgage and you're stuck. Heads you lose a little, tails you lose a little: that's **negative convexity**, and the **spread** the mortgage pays over a Treasury is your rent for wearing it. The carry trade: buy the mortgage bond, short just enough Treasury to cancel the interest-rate move, and keep the spread.

In [1]:
import numpy as np, pandas as pd
R = dict(mbb_carry=0.3, mbb_t=0.64, mbb_ci_lo=-0.62, mbb_ci_hi=1.2, race_adv=0.016,
         race_mbs_sh=0.336, race_ief_sh=0.32, net_mbb=-0.16)
print('duration-neutral MBS carry (MBB): %+.2f%%/yr  (HAC t = %+.2f)'
      % (R['mbb_carry'], R['mbb_t']))
print('  bootstrap 95%% CI: [%+.2f, %+.2f] %%/yr  -> straddles zero'
      % (R['mbb_ci_lo'], R['mbb_ci_hi']))
print('  Sharpe advantage over duration-matched IEF: %+.3f  (a tie)' % R['race_adv'])
print('  net carry after costs: %+.2f%%/yr  -> below zero' % R['net_mbb'])

duration-neutral MBS carry (MBB): +0.30%/yr  (HAC t = +0.64)
  bootstrap 95% CI: [-0.62, +1.20] %/yr  -> straddles zero
  Sharpe advantage over duration-matched IEF: +0.016  (a tie)
  net carry after costs: -0.16%/yr  -> below zero


## 2. Where did the spread go?

The option-adjusted spread is *real* — but you get paid it in calm markets and it is **clawed straight back** in the rate shocks the convexity exposes you to. The one great year was **2009** (+8.2%, post-crash spread compression, a one-off); the deep holes are the rate shocks **2008, 2011, and 2022** (-4.4%). Across the full cycle it nets to about **+0.30%/yr** — indistinguishable from zero, and *negative* once you pay trading costs.

## 3. Is the machine honest? A live synthetic control

We plant a known +2%/yr carry in a seeded toy world (a shared rate factor drives both legs) and check the duration-neutral estimator recovers it — and that it stays **silent** when there is no carry to find. No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from mbs_carry import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(carry_annual=0.0, seed=886))
planted = st.synthetic_detect(data.synthetic_world(carry_annual=0.02, seed=886))
print('null world   : carry %+.2f%%/yr  HAC t = %+.2f  (should be ~0)'
      % (null['carry_ann_pct'], null['t_hac']))
print('planted +2%%  : carry %+.2f%%/yr  HAC t = %+.2f  (should light up)'
      % (planted['carry_ann_pct'], planted['t_hac']))

null world   : carry +0.26%/yr  HAC t = +0.67  (should be ~0)
planted +2%  : carry +2.26%/yr  HAC t = +5.95  (should light up)


## 4. The honest verdict

On the live ETF tape the duration-neutral agency-MBS carry is **+0.30%/yr at HAC *t* = +0.64** — the right sign, but the bootstrap CI **[-0.62, +1.20]** straddles zero, the Sharpe edge over duration-matched IEF is **+0.016** (a tie), the carry **collapses to +0.17%/yr (*t* +0.18)** in the 2020-2026 rate-vol era, and the net goes **-0.16%/yr after costs**. The premium is real ex-ante; negative convexity eats it. **Signal: Weak · Tradability: Mirage.**